

## 📊 Sentiment Polarity Extraction

### Purpose and Rationale

This notebook computes **sentiment polarity scores** from social media and online discourse related to H5N1 outbreaks. Sentiment polarity serves as a **continuous affective signal** that complements categorical stigma labels (fear, urgency, humor, neutral) by capturing **gradual shifts in emotional valence** over time.

Unlike discrete stigma classification, polarity provides a **directional and intensity-aware measure** of public concern, enabling the model to detect subtle emotional build-up that may precede outbreak confirmations.

---

### Method Overview

We use the **VADER (Valence Aware Dictionary and sEntiment Reasoner)** sentiment analyzer from NLTK, which is specifically designed for short, informal text such as social media posts and news headlines.

For each document:

1. Text is first **cleaned and normalized** to ensure compatibility with the sentiment model.
2. VADER computes four sentiment scores:

   * **Positive**
   * **Negative**
   * **Neutral**
   * **Compound** (normalized, weighted summary score)
3. The **compound score** (range: −1 to +1) is retained as the primary sentiment polarity signal.

---

### Why VADER?

VADER was selected because it:

* Performs well on **short, noisy, real-world text**
* Is **lexicon-based and deterministic**, improving reproducibility
* Does not require task-specific training data
* Captures **negation, intensifiers, and punctuation effects**

This makes it particularly suitable for large-scale epidemiological discourse analysis where labeled sentiment data is unavailable.

---

### Polarity Interpretation

| Polarity Range | Interpretation                                   |
| -------------- | ------------------------------------------------ |
| −1.0 to −0.3   | Strong negative sentiment (fear, alarm, concern) |
| −0.3 to 0.0    | Mild negative sentiment                          |
| 0.0 to +0.3    | Mild positive / neutral sentiment                |
| +0.3 to +1.0   | Strong positive sentiment                        |

In downstream modeling, **negative polarity values** often co-occur with fear- and urgency-related stigma, while neutral or positive values frequently reflect informational or reassurance-focused discourse.

---

### Role in the Final Model

Sentiment polarity is used as:

* A **time-varying continuous feature**
* A complementary signal to binary stigma indicators
* An input to **lead–lag feature engineering**, including:

  * Lagged polarity values
  * Rolling averages (emotional persistence)

This enables the model to distinguish **short-lived emotional spikes** from **sustained affective escalation**, which is critical for early-warning detection.

---

## Optional: Per-Cell Markdown Comments

### Cell: Load libraries and initialize VADER

```markdown
This cell imports required libraries and initializes the VADER sentiment analyzer. The lexicon is downloaded once and reused to ensure consistent sentiment scoring across the corpus.
```

### Cell: Apply sentiment scoring

```markdown
This cell applies VADER sentiment analysis to each cleaned text entry and extracts the compound polarity score, which serves as the primary sentiment feature in subsequent modeling.
```

### Cell: Save enriched dataset

```markdown
This cell persists the dataset with appended sentiment polarity scores for downstream feature engineering and modeling stages.
```



In [9]:
import pandas as pd
import re
import nltk
import pandas as pd
import numpy as np
import re
from datetime import datetime
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from textblob import TextBlob
import matplotlib.pyplot as plt
# download resources once
nltk.download('vader_lexicon')
from nltk.sentiment.vader import SentimentIntensityAnalyzer

sid = SentimentIntensityAnalyzer()

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/gazimahmud/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [15]:
# Apply to your dataset
df = pd.read_csv("merged_output_with_stigma_ohe.csv")
df["text"].head()


0    Despite Owners Exhausting Legal Appeals, The C...
1    Convorbiri literare 1. Fata noastra lasconista...
2    Is this the face of evil? Apparently, this man...
3    A case of Highly Pathogenic Avian Influenza (H...
4    Diese Symptome zeigen Katzen bei Vogelgrippe\n...
Name: text, dtype: object

In [16]:
df.dtypes

activities                              float64
content_type                             object
creation_time                            object
id                                        int64
is_branded_content                         bool
lang                                     object
link_attachment.caption                  object
link_attachment.description              object
link_attachment.link                     object
link_attachment.name                     object
match_type                               object
mcl_url                                  object
modified_time                            object
multimedia                               object
post_owner.id                            object
post_owner.name                          object
post_owner.type                          object
post_owner.username                      object
shared_post_id                          float64
statistics.angry_count                  float64
statistics.care_count                   

In [18]:
# sentiment polarity (-1 to 1)
df["clean_text"] = df["clean_text"].fillna("")
df['sentiment'] = df['clean_text'].apply(lambda x: TextBlob(x).sentiment.polarity)
df.columns

Index(['activities', 'content_type', 'creation_time', 'id',
       'is_branded_content', 'lang', 'link_attachment.caption',
       'link_attachment.description', 'link_attachment.link',
       'link_attachment.name', 'match_type', 'mcl_url', 'modified_time',
       'multimedia', 'post_owner.id', 'post_owner.name', 'post_owner.type',
       'post_owner.username', 'shared_post_id', 'statistics.angry_count',
       'statistics.care_count', 'statistics.comment_count',
       'statistics.haha_count', 'statistics.like_count',
       'statistics.love_count', 'statistics.reaction_count',
       'statistics.sad_count', 'statistics.share_count', 'statistics.views',
       'statistics.views_date_last_refreshed', 'statistics.wow_count',
       'surface.id', 'surface.name', 'surface.type', 'surface.username',
       'text', 'clean_text', 'inferred_country', 'inferred_country_confidence',
       'stigma_label', 'stigma_fear', 'stigma_humor', 'stigma_neutral',
       'stigma_urgency', 'stigma_fear.1'

In [19]:
df.head(10)

,activities,content_type,creation_time,id,is_branded_content,lang,link_attachment.caption,link_attachment.description,link_attachment.link,link_attachment.name,...,stigma_label,stigma_fear,stigma_humor,stigma_neutral,stigma_urgency,stigma_fear.1,stigma_humor.1,stigma_neutral.1,stigma_urgency.1,sentiment
0,NaN,status,2025-11-10T19:00:19+00:00,2980185815499999,False,en,NaN,NaN,NaN,NaN,...,fear,True,False,False,False,True,False,False,False,-0.243182
1,NaN,albums,2025-11-10T18:23:30+00:00,686591307858179,False,ro,NaN,NaN,NaN,NaN,...,fear,True,False,False,False,True,False,False,False,0.000000
2,NaN,videos,2025-11-10T18:14:12+00:00,1006177595025072,False,en,NaN,NaN,NaN,NaN,...,fear,True,False,False,False,True,False,False,False,-0.075000
3,NaN,links,2025-11-10T16:24:10+00:00,840171035327111,False,en,brecon-radnor.co.uk,NaN,https://www.brecon-radnor.co.uk/news/farming/a...,Avian influenza confirmed at Powys premises,...,neutral,False,False,True,False,False,False,True,False,0.280000
4,NaN,links,2025-11-10T16:05:06+00:00,1028189766103347,False,de,promisundmehr.de,"Promis, Prominente, Stars und Sternchen ... Di...",https://www.promisundmehr.de/vogelgrippe-bei-h...,Vogelgrippe bei Haustieren: Katzen sind potenz...,...,fear,True,False,False,False,True,False,False,False,0.000000
5,NaN,photos,2025-11-10T15:55:13+00:00,702937055735088,False,en,NaN,NaN,NaN,NaN,...,fear,True,False,False,False,True,False,False,False,0.095920
6,NaN,videos,2025-11-10T15:36:02+00:00,698739099514974,False,en,NaN,NaN,NaN,NaN,...,urgency,False,False,False,True,False,False,False,True,0.109505
7,NaN,videos,2025-11-10T15:31:47+00:00,1218604383464251,False,en,NaN,NaN,NaN,NaN,...,urgency,False,False,False,True,False,False,False,True,0.109505
8,NaN,links,2025-11-10T15:29:50+00:00,770556289362124,False,en,cheknews.ca,An animal sanctuary in B.C.'s interior says it...,https://cheknews.ca/animal-sanctuary-staff-in-...,"Animal sanctuary staff in Summerland, B.C., 'd...",...,neutral,False,False,True,False,False,False,True,False,0.227273
9,NaN,status,2025-11-10T15:24:36+00:00,1367013008148200,False,de,NaN,NaN,NaN,NaN,...,fear,True,False,False,False,True,False,False,False,0.000000


In [20]:
#OneHotEncoded
df.to_csv("merged_output_with_stigma_ohe_sentiment.csv", index=False)